# Step By Step 03 - 因子设计

Compose, evaluate and serialize temporal and cross-sectional formulas.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.data.transformer import Transformer
from alphamind.formula import LAST, MA, SHIFT, CSQuantiles, CSRank, decode_formula, encode_formula

expressions = {
    "raw": LAST("factor"),
    "lag": SHIFT("factor", 1),
    "mean": MA(3, "factor"),
    "rank": CSRank("factor"),
    "percentile": CSQuantiles("factor"),
    "combined": LAST("factor") + 0.5 * LAST("value"),
}
result = await asyncio.to_thread(Transformer(expressions).transform, "code", panel)
restored = decode_formula(encode_formula(expressions["lag"]))
restored_result = await asyncio.to_thread(Transformer({"lag": restored}).transform, "code", panel)
assert result["lag"].to_pylist() == restored_result["lag"].to_pylist()
result
